In [2]:
%pip install -q langchain langgraph langchain-groq pydantic python-dotenv duckduckgo-search ddgs
import os
import json
import requests
from typing import Annotated, TypedDict
from dotenv import load_dotenv

# LangChain Core & Groq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

# LangGraph Core Components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in your .env file")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


1. STATE DEFINITION

In [3]:
class AgentState(TypedDict):
    """
    The graph state maintains conversation context across nodes.
    The `add_messages` reducer ensures new messages are appended rather than overwritten.
    """
    messages: Annotated[list[BaseMessage], add_messages]

2. DEFINE TOOLS

In [4]:
class SearchInput(BaseModel):
    query: str = Field(description="The web search query keywords to search for.")

@tool(args_schema=SearchInput)
def web_search(query: str) -> str:
    """Search the internet for real-time information, news, weather, or facts."""
    from duckduckgo_search import DDGS
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No search results found."
        formatted = [
            f"Title: {r.get('title')}\nSnippet: {r.get('body')}\nURL: {r.get('href')}"
            for r in results
        ]
        return "\n---\n".join(formatted)
    except Exception as e:
        return f"Error running search: {str(e)}"

@tool
def get_weather(city: str) -> str:
    """Fetch current weather using Open-Meteo or wttr.in."""
    res = requests.get(f"https://wttr.in/{city}?format=3")
    return res.text

tools = [web_search, get_weather]

3. MODELS & FALLBACK LOGIC

In [5]:
primary_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
fallback_llm = ChatGroq(model="qwen/qwen3.6-27b", temperature=0.0)

# Bind tools so the models can generate tool calls
primary_model = primary_llm.bind_tools(tools)
fallback_model = fallback_llm.bind_tools(tools)

4. NODES (Graph Processing Units)

In [6]:
def safety_guard_node(state: AgentState):
    """
    Node 1: Validates incoming queries for topic safety.
    """
    messages = state["messages"]
    if not messages:
        return {}

    first_msg = messages[0]
    content = first_msg.content if hasattr(first_msg, "content") else str(first_msg)

    # Fast evaluation using classifier prompt
    classification = primary_llm.invoke([
        SystemMessage(content="You are a safety filter. Respond ONLY in JSON:\n{\"safe\": true} or {\"safe\": false}"),
        HumanMessage(content=content)
    ])

    try:
        result = json.loads(classification.content)
        is_safe = result.get("safe", True)
    except Exception:
        is_safe = True

    if not is_safe:
        return {
            "messages": [AIMessage(content="❌ Request blocked due to safety guidelines.")]
        }
    return {}


def agent_node(state: AgentState):
    """
    Node 2: Invokes the primary LLM with fallback support.
    """
    sys_prompt = SystemMessage(
        content=(
            "You are a specialized Assistant. "
            "Help users with weather, real-time facts, and web searches using tools when needed."
        )
    )
    full_messages = [sys_prompt] + state["messages"]

    try:
        response = primary_model.invoke(full_messages)
    except Exception as e:
        print(f"⚠️ Primary model error: {e}. Executing fallback model...")
        response = fallback_model.invoke(full_messages)

    return {"messages": [response]}


def safety_router(state: AgentState):
    """
    Conditional edge decision: If safety filter injected a block message, exit early.
    """
    last_message = state["messages"][-1] if state["messages"] else None
    if last_message and "Request blocked" in str(last_message.content):
        return END
    return "agent"


# Node 3: Prebuilt Tool execution node
tool_node = ToolNode(tools)

5. GRAPH CONSTRUCTION & WORKFLOW

In [7]:
builder = StateGraph(AgentState)

# Add Nodes to Graph
builder.add_node("safety_guard", safety_guard_node)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

# Flow Setup (Edges)
builder.add_edge(START, "safety_guard")

# Conditional Edge 1: Stop execution if safety fails, otherwise proceed to agent
builder.add_conditional_edges("safety_guard", safety_router, ["agent", END])

# Conditional Edge 2: Agent execution decisions (Routes to 'tools' or END)
builder.add_conditional_edges("agent", tools_condition)

# Normal Edge: Return output from tool execution back to the agent for evaluation
builder.add_edge("tools", "agent")

# Compile with Persistence Checkpointer
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

6. RUNNING THE WORKFLOW

In [8]:
if __name__ == "__main__":
    from langchain_core.utils.uuid import uuid7
    
    thread_config = {"configurable": {"thread_id": str(uuid7())}}
    
    print("--- Query 1: Weather Check ---")
    inputs = {"messages": [HumanMessage(content="What is the current weather in Cairo right now?")]}
    
    for event in graph.stream(inputs, config=thread_config, stream_mode="values"):
        last_msg = event["messages"][-1]
        print(f"[{last_msg.type.upper()}]: {last_msg.content}")
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            print(f"  -> Tool Calls: {last_msg.tool_calls}")

--- Query 1: Weather Check ---
[HUMAN]: What is the current weather in Cairo right now?
[AI]: 
  -> Tool Calls: [{'name': 'get_weather', 'args': {'city': 'Cairo'}, 'id': 'fc_96735ce3-64dc-4970-8f5d-c46984188be7', 'type': 'tool_call'}]
[TOOL]: Cairo: ☀️  +33°C

[AI]: **Current weather in Cairo**

- ☀️ **Sunny**  
- Temperature: **+33 °C**  

Let me know if you’d like more details (humidity, wind, forecast, etc.)!
